<a href="https://colab.research.google.com/github/asfiya-tehmeen/Clinical-Trial-Patient-Matching-Agent/blob/main/ClinicalTrialMatching_git.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# ============================================================
# CLINICAL TRIAL PATIENT MATCHING AGENT
# ============================================================

# CELL 1 — Install dependencies

!pip install -q openai chromadb requests pandas pydantic rich

print("✅ Dependencies installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [10]:
# CELL 2 — GROQ API Key setup

from openai import OpenAI

GROQ_API_KEY = "gsk....."   # ← your Groq key

client = OpenAI(
    api_key  = GROQ_API_KEY,
    base_url = "https://api.groq.com/openai/v1"
)

print("✅ Groq client ready")
print("   Base URL : https://api.groq.com/openai/v1")

✅ Groq client ready
   Base URL : https://api.groq.com/openai/v1


In [11]:
# CELL 3 — Imports & config

import json
import time
import requests
import chromadb
import pandas as pd
from pydantic import BaseModel
from typing import List, Optional, Literal
from chromadb.utils import embedding_functions

# Model config
# Use grok-3-mini for fast/cheap steps (extraction, summaries)
# Use grok-3 for deep reasoning steps (criteria analysis)
MODEL_FAST      = "llama-3.1-8b-instant"
MODEL_REASONING = "llama-3.3-70b-versatile"

CTGOV_URL  = "https://clinicaltrials.gov/api/v2/studies"
MAX_TRIALS = 25    # trials fetched before LLM reasoning
TOP_K      = 5     # top matches shown in final output

print("✅ All imports ready")
print(f"   Fast model      : {MODEL_FAST}")
print(f"   Reasoning model : {MODEL_REASONING}")
print(f"   Trials fetched  : {MAX_TRIALS}")
print(f"   Top-K shown     : {TOP_K}")

✅ All imports ready
   Fast model      : llama-3.1-8b-instant
   Reasoning model : llama-3.3-70b-versatile
   Trials fetched  : 25
   Top-K shown     : 5


In [12]:
# CELL 4 — Synthetic patient (edit this to test your own cases)

PATIENT_EHR = """
Patient: 61-year-old female
Primary diagnosis: Stage IIIB non-small cell lung cancer (NSCLC),
  adenocarcinoma histology, confirmed by biopsy 4 months ago.
Molecular: EGFR exon 19 deletion positive. ALK negative. PD-L1 TPS 45%.
Performance status: ECOG 1
Comorbidities: Type 2 diabetes (well-controlled, HbA1c 7.1%),
  mild hypertension on lisinopril 10mg daily.
Current medications: Metformin 1000mg BID, Lisinopril 10mg QD,
  Ondansetron PRN.
Prior treatments: Completed 4 cycles carboplatin + pemetrexed (first-line).
  No prior EGFR-targeted therapy. No prior immunotherapy.
Recent labs: eGFR 68 mL/min, ALT 32 U/L, ANC 1.9 x10^9/L,
  Hgb 11.2 g/dL, platelets 210 x10^9/L.
Brain MRI: No intracranial metastases detected.
Location: Boston, MA, USA
"""

print("✅ Patient profile loaded")
print("─" * 50)
print(PATIENT_EHR.strip())

✅ Patient profile loaded
──────────────────────────────────────────────────
Patient: 61-year-old female
Primary diagnosis: Stage IIIB non-small cell lung cancer (NSCLC),
  adenocarcinoma histology, confirmed by biopsy 4 months ago.
Molecular: EGFR exon 19 deletion positive. ALK negative. PD-L1 TPS 45%.
Performance status: ECOG 1
Comorbidities: Type 2 diabetes (well-controlled, HbA1c 7.1%),
  mild hypertension on lisinopril 10mg daily.
Current medications: Metformin 1000mg BID, Lisinopril 10mg QD,
  Ondansetron PRN.
Prior treatments: Completed 4 cycles carboplatin + pemetrexed (first-line).
  No prior EGFR-targeted therapy. No prior immunotherapy.
Recent labs: eGFR 68 mL/min, ALT 32 U/L, ANC 1.9 x10^9/L,
  Hgb 11.2 g/dL, platelets 210 x10^9/L.
Brain MRI: No intracranial metastases detected.
Location: Boston, MA, USA


In [13]:
# TEST CELL — Verify Groq is working
print("Testing Groq connection...")

test = client.chat.completions.create(
    model      = "llama-3.1-8b-instant",
    max_tokens = 50,
    messages   = [{"role": "user", "content": "Say hello in one word."}]
)

print("✅ Groq works! Response:", test.choices[0].message.content)
print("   Model used:", test.model)

Testing Groq connection...
✅ Groq works! Response: Hello.
   Model used: llama-3.1-8b-instant


In [14]:
# CELL 5 — EHR extraction agent

# Converts free-text clinical note → structured JSON profile

EXTRACTION_PROMPT = """
You are a clinical NLP specialist. Extract structured information from
this clinical note. Return ONLY valid JSON — no preamble, no markdown fences.

Extract exactly this schema:
{{
  "demographics": {{
    "age": <int>,
    "sex": "<M or F>",
    "location": "<city, state, country>"
  }},
  "primary_diagnosis": {{
    "name": "<full diagnosis name>",
    "icd10_guess": "<best ICD-10 code>",
    "stage": "<stage if available>",
    "histology": "<histology if available>"
  }},
  "biomarkers": [
    {{"name": "<biomarker>", "value": "<result>", "status": "<positive/negative/numeric>"}}
  ],
  "medications": [
    {{"name": "<drug name>", "dose": "<dose>", "frequency": "<frequency>"}}
  ],
  "labs": [
    {{"name": "<lab name>", "value": "<value>", "unit": "<unit>"}}
  ],
  "ecog_status": <int 0-4>,
  "prior_treatments": ["<treatment 1>", "<treatment 2>"],
  "comorbidities": ["<condition 1>", "<condition 2>"],
  "exclusion_flags": ["<anything that commonly excludes patients from trials>"],
  "key_search_terms": ["<3-6 best search terms for finding relevant trials>"]
}}

Clinical note:
\"\"\"
{ehr_text}
\"\"\"
"""

def call_grok(prompt: str, model: str, max_tokens: int = 1500) -> str:
    """Unified Grok API call — returns response text."""
    response = client.chat.completions.create(
        model      = model,
        max_tokens = max_tokens,
        messages   = [{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip()

def clean_json(raw: str) -> dict:
    """Strip markdown fences if model wraps output in them, then parse JSON."""
    if raw.startswith("```"):
        parts = raw.split("```")
        raw = parts[1] if len(parts) > 1 else raw
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())

def extract_patient_profile(ehr_text: str) -> dict:
    print("🔍 Running EHR extraction with grok-3-mini...")
    raw = call_grok(EXTRACTION_PROMPT.format(ehr_text=ehr_text), "llama-3.1-8b-instant")
    profile = clean_json(raw)
    print("🔍 Running EHR extraction with llama-3.1-8b-instant (Groq)...")
    return profile

patient_profile = extract_patient_profile(PATIENT_EHR)
print("\n📋 Structured patient profile:")
print(json.dumps(patient_profile, indent=2))



🔍 Running EHR extraction with grok-3-mini...
🔍 Running EHR extraction with llama-3.1-8b-instant (Groq)...

📋 Structured patient profile:
{
  "demographics": {
    "age": 61,
    "sex": "F",
    "location": "Boston, MA, USA"
  },
  "primary_diagnosis": {
    "name": "Stage IIIB non-small cell lung cancer",
    "icd10_guess": "C34.92",
    "stage": "IIIB",
    "histology": "adenocarcinoma"
  },
  "biomarkers": [
    {
      "name": "EGFR exon 19 deletion",
      "value": "positive",
      "status": "positive"
    },
    {
      "name": "ALK",
      "value": "negative",
      "status": "negative"
    },
    {
      "name": "PD-L1 TPS",
      "value": "45%",
      "status": "numeric"
    }
  ],
  "medications": [
    {
      "name": "metformin",
      "dose": "1000mg",
      "frequency": "BID"
    },
    {
      "name": "lisinopril",
      "dose": "10mg",
      "frequency": "QD"
    },
    {
      "name": "ondansetron",
      "dose": "PRN",
      "frequency": "PRN"
    }
  ],
  "labs": [
 

In [16]:
# CELL 6 — ClinicalTrials.gov search

# Queries the free API with multiple search strategies — no key needed

def search_clinicaltrials(query_terms: list, condition: str, max_results: int = MAX_TRIALS) -> list:
    """Query ClinicalTrials.gov v2 API with multiple strategies, deduplicate results."""
    all_trials = {}
    search_queries = [condition] + query_terms[:3]

    for query in search_queries:
        params = {
            "query.cond"          : query,
            "filter.overallStatus": "RECRUITING",
            "fields"              : ",".join([
                "NCTId", "BriefTitle", "OfficialTitle",
                "EligibilityCriteria", "OverallStatus",
                "Phase", "LocationCity", "LocationCountry",
                "LeadSponsorName", "BriefSummary",
                "MinimumAge", "MaximumAge", "Sex",
                "PointOfContactOrganization", "PointOfContactEMail"
            ]),
            "pageSize"   : 15,
            "countTotal" : "true"
        }
        try:
            resp = requests.get(CTGOV_URL, params=params, timeout=15)
            resp.raise_for_status()
            data    = resp.json()
            studies = data.get("studies", [])
            for s in studies:
                proto  = s.get("protocolSection", {})
                id_mod = proto.get("identificationModule", {})
                nct_id = id_mod.get("nctId", "")
                if nct_id and nct_id not in all_trials:
                    all_trials[nct_id] = proto
            print(f"   Query '{query[:45]}' → {len(studies)} trials")
        except Exception as e:
            print(f"   ⚠ Query failed: {e}")
        time.sleep(0.3)

    print(f"✅ Total unique trials fetched: {len(all_trials)}")
    return list(all_trials.values())[:max_results]

condition    = patient_profile["primary_diagnosis"]["name"]
search_terms = patient_profile.get("key_search_terms", [])

print(f"\nSearching trials for: {condition}")
print(f"   Additional terms: {search_terms}")
raw_trials = search_clinicaltrials(search_terms, condition)


Searching trials for: Stage IIIB non-small cell lung cancer
   Additional terms: ['Stage IIIB NSCLC', 'adenocarcinoma', 'EGFR exon 19 deletion', 'non-small cell lung cancer']
   Query 'Stage IIIB non-small cell lung cancer' → 15 trials
   Query 'Stage IIIB NSCLC' → 15 trials
   Query 'adenocarcinoma' → 15 trials
   Query 'EGFR exon 19 deletion' → 5 trials
✅ Total unique trials fetched: 35


In [17]:
# CELL 7 — Parse trials into clean dicts

def parse_trial(proto: dict) -> dict:
    """Flatten a ClinicalTrials.gov protocolSection into a clean usable dict."""
    id_mod      = proto.get("identificationModule", {})
    status_mod  = proto.get("statusModule", {})
    desc_mod    = proto.get("descriptionModule", {})
    elig_mod    = proto.get("eligibilityModule", {})
    design_mod  = proto.get("designModule", {})
    sponsor_mod = proto.get("sponsorCollaboratorsModule", {})
    contact_mod = proto.get("contactsLocationsModule", {})

    locations = contact_mod.get("locations", [])
    cities = list({
        loc.get("city", "") + ", " + loc.get("country", "")
        for loc in locations[:5] if loc.get("city")
    })

    phases = design_mod.get("phases", ["N/A"])
    phase  = phases[0] if phases else "N/A"

    return {
        "nct_id"          : id_mod.get("nctId", ""),
        "title"           : id_mod.get("briefTitle", ""),
        "official_title"  : id_mod.get("officialTitle", ""),
        "status"          : status_mod.get("overallStatus", ""),
        "phase"           : phase,
        "sponsor"         : sponsor_mod.get("leadSponsor", {}).get("name", ""),
        "brief_summary"   : desc_mod.get("briefSummary", "")[:500],
        "eligibility_text": elig_mod.get("eligibilityCriteria", "")[:3000],
        "min_age"         : elig_mod.get("minimumAge", ""),
        "max_age"         : elig_mod.get("maximumAge", ""),
        "sex"             : elig_mod.get("sex", "ALL"),
        "sites"           : cities,
        "url"             : f"https://clinicaltrials.gov/study/{id_mod.get('nctId', '')}"
    }

parsed_trials = [parse_trial(t) for t in raw_trials]
parsed_trials = [t for t in parsed_trials if t["eligibility_text"]]

print(f"✅ Parsed {len(parsed_trials)} trials with eligibility text")
df_preview = pd.DataFrame(parsed_trials)[["nct_id","title","phase","sponsor"]].head(8)
print(df_preview.to_string(index=False))


✅ Parsed 25 trials with eligibility text
     nct_id                                                                                                                                                                                      title  phase                                               sponsor
NCT06734182                                                                        Neoadjuvant Envafolimab Plus Disitamab Vedotin and Carboplatin in Resectable HER2-Mutant Non-Small-Cell Lung Cancer PHASE2                Guangdong Provincial People's Hospital
NCT04267848            Testing the Addition of a Type of Drug Called Immunotherapy to the Usual Chemotherapy Treatment for Non-small Cell Lung Cancer, an ALCHEMIST Treatment Trial (Chemo-IO [ACCIO]) PHASE3                       National Cancer Institute (NCI)
NCT05061550                                                                                                                Neoadjuvant and Adjuvant Treatment in Resectable Non-sma

In [21]:
# CELL 8 — Hard pre-filter


def age_passes(age: int, min_age_str: str, max_age_str: str) -> bool:
    def parse_age(s):
        if not s: return None
        digits = ''.join(filter(str.isdigit, s.split()[0]))
        return int(digits) if digits else None
    lo = parse_age(min_age_str)
    hi = parse_age(max_age_str)
    if lo and age < lo: return False
    if hi and age > hi: return False
    return True

def sex_passes(patient_sex: str, trial_sex: str) -> bool:
    if trial_sex.upper() in ("ALL", ""): return True
    return patient_sex.upper()[0] == trial_sex.upper()[0]

patient_age = patient_profile["demographics"]["age"]
patient_sex = patient_profile["demographics"]["sex"]

pre_filtered = []
removed      = []
for trial in parsed_trials:
    age_ok = age_passes(patient_age, trial["min_age"], trial["max_age"])
    sex_ok = sex_passes(patient_sex, trial["sex"])
    if age_ok and sex_ok:
        pre_filtered.append(trial)
    else:
        reason = []
        if not age_ok: reason.append(f"age {patient_age} outside {trial['min_age']}–{trial['max_age']}")
        if not sex_ok: reason.append(f"sex {patient_sex} vs {trial['sex']}")
        removed.append((trial["nct_id"], ", ".join(reason)))

print(f"✅ Pre-filter: {len(parsed_trials)} → {len(pre_filtered)} trials")
if removed:
    print(f"   Removed {len(removed)} trials:")
    for nct, reason in removed[:5]:
        print(f"   ✗ {nct}: {reason}")


✅ Pre-filter: 25 → 24 trials
   Removed 1 trials:
   ✗ NCT07552922: sex F vs MALE


In [23]:
# CELL 9 — Criteria reasoning agent (the core — uses grok-3)

# For each trial, Grok checks every inclusion/exclusion criterion.
# Each criterion gets: PASS / FAIL / UNCERTAIN / MISSING_DATA

CRITERIA_PROMPT = """
You are a clinical trial eligibility specialist with deep knowledge of
oncology trial protocols.

PATIENT PROFILE:
{patient_json}

CLINICAL TRIAL: {nct_id}
Title: {title}

ELIGIBILITY CRITERIA:
{eligibility_text}

For each inclusion and exclusion criterion, determine if this patient meets it.
Return ONLY valid JSON — no preamble, no markdown fences.

{{
  "trial_nct_id": "{nct_id}",
  "overall_eligible": true | false | "uncertain",
  "criteria_results": [
    {{
      "criterion_text": "<exact criterion text, shortened to 80 chars>",
      "type": "inclusion" | "exclusion",
      "verdict": "PASS" | "FAIL" | "UNCERTAIN" | "MISSING_DATA",
      "confidence": <float 0.0-1.0>,
      "reasoning": "<one sentence citing specific patient data>",
      "missing_info": "<if MISSING_DATA: what test or info resolves this, else null>"
    }}
  ],
  "disqualifying_criteria": ["<criterion text if FAIL>"],
  "uncertain_criteria": ["<criterion text if UNCERTAIN>"],
  "missing_data_requests": ["<actionable test or data needed, if any>"],
  "match_summary": "<2 sentence plain-English summary for a physician>"
}}

Rules:
- PASS         = patient clearly meets this criterion based on available data
- FAIL         = patient clearly does NOT meet this criterion (disqualifying)
- UNCERTAIN    = ambiguous — not enough info to decide either way
- MISSING_DATA = patient may qualify but a specific piece of data is absent
- If ANY inclusion criterion is FAIL → overall_eligible = false
- If ANY exclusion criterion is PASS (patient HAS the exclusion) → overall_eligible = false
- Be specific — cite actual values (e.g. "ECOG 1 meets PS 0-2 requirement")
"""

def reason_over_trial(trial: dict, patient_profile: dict) -> dict:
    """Run Grok-3 reasoning over a single trial's eligibility criteria."""
    prompt = CRITERIA_PROMPT.format(
        patient_json     = json.dumps(patient_profile, indent=2),
        nct_id           = trial["nct_id"],
        title            = trial["title"],
        eligibility_text = trial["eligibility_text"][:2500]
    )
    try:
        raw = call_grok(prompt, "llama-3.3-70b-versatile", max_tokens=2000)
        result = clean_json(raw)
        result["trial_title"] = trial["title"]
        result["trial_phase"] = trial["phase"]
        result["trial_url"]   = trial["url"]
        result["trial_sites"] = trial["sites"]
        result["sponsor"]     = trial["sponsor"]
        return result
    except Exception as e:
        return {
            "trial_nct_id"         : trial["nct_id"],
            "trial_title"          : trial["title"],
            "overall_eligible"     : False,
            "criteria_results"     : [],
            "match_summary"        : f"Parsing error: {e}",
            "missing_data_requests": [],
            "error"                : str(e)
        }

print(f" Running grok-3 reasoning on {len(pre_filtered)} trials...")
print("   (~5–10 sec per trial — grab a coffee)\n")

all_results = []
for i, trial in enumerate(pre_filtered):
    print(f"   [{i+1}/{len(pre_filtered)}] {trial['nct_id']} — {trial['title'][:55]}...")
    result = reason_over_trial(trial, patient_profile)
    all_results.append(result)
    time.sleep(0.5)

print(f"\n✅ Reasoning complete on {len(all_results)} trials")

 Running grok-3 reasoning on 24 trials...
   (~5–10 sec per trial — grab a coffee)

   [1/24] NCT06734182 — Neoadjuvant Envafolimab Plus Disitamab Vedotin and Carb...
   [2/24] NCT04267848 — Testing the Addition of a Type of Drug Called Immunothe...
   [3/24] NCT05061550 — Neoadjuvant and Adjuvant Treatment in Resectable Non-sm...
   [4/24] NCT05341583 — Ensartinib as Adjuvant Treatment in Anaplastic Lymphoma...
   [5/24] NCT05989542 — A Confirmatory Clinical Study in NSCLC Patients With ME...
   [6/24] NCT06269211 — Neoadjuvant Toripalimab for Clinically Stage II-IIIB Re...
   [7/24] NCT07534033 — Phase II Clinical Study of Probiotic LR607 in Patients ...
   [8/24] NCT07288034 — Immunotherapy Biomarkers to Predict First-line PD(L)1-b...
   [9/24] NCT06951464 — A Study of BL-B01D1 and Almonertinib in Patients With R...
   [10/24] NCT07003490 — Radical Resection With Contralateral Lymph Node Dissect...
   [11/24] NCT06788912 — Pembrolizumab (MK-3475) Plus Investigational Agents in ...
 

In [24]:
# CELL 10 — Scoring and ranking

def score_trial(result: dict) -> float:
    """
    Composite match score (0.0 – 1.0).
    Any hard FAIL → 0.0. Otherwise: pass rate minus uncertainty penalty + phase bonus.
    """
    if result.get("overall_eligible") == False:
        return 0.0

    criteria = result.get("criteria_results", [])
    if not criteria:
        return 0.0

    total     = len(criteria)
    passes    = sum(1 for c in criteria if c["verdict"] == "PASS")
    fails     = sum(1 for c in criteria if c["verdict"] == "FAIL")
    uncertain = sum(1 for c in criteria if c["verdict"] in ("UNCERTAIN", "MISSING_DATA"))

    if fails > 0:
        return 0.0

    pass_rate       = passes / total
    uncertainty_pen = (uncertain / total) * 0.25

    phase_map  = {"PHASE3": 0.10, "PHASE2": 0.05, "PHASE1": 0.0, "N/A": 0.0}
    phase_str  = result.get("trial_phase", "N/A").replace(" ", "").upper()
    phase_bonus= phase_map.get(phase_str, 0.0)

    return round(min(1.0, pass_rate - uncertainty_pen + phase_bonus), 3)

for r in all_results:
    r["match_score"] = score_trial(r)

eligible   = sorted([r for r in all_results if r["match_score"] > 0],
                    key=lambda x: x["match_score"], reverse=True)
ineligible = [r for r in all_results if r["match_score"] == 0]

print(f"✅ Scoring complete")
print(f"   Eligible / uncertain : {len(eligible)}")
print(f"   Disqualified         : {len(ineligible)}")



✅ Scoring complete
   Eligible / uncertain : 3
   Disqualified         : 21


In [25]:
# CELL 11 — Display ranked results

def verdict_icon(v):
    return {"PASS": "✅", "FAIL": "❌", "UNCERTAIN": "⚠️ ", "MISSING_DATA": "🔬"}.get(v, "❓")

def print_match_card(result: dict, rank: int):
    score = result["match_score"]
    bar   = "█" * int(score * 10) + "░" * (10 - int(score * 10))

    print(f"\n{'='*65}")
    print(f"  MATCH #{rank}  |  Score: {score:.0%}  [{bar}]")
    print(f"{'='*65}")
    print(f"  📋 {result['trial_title']}")
    print(f"  🔑 {result.get('trial_nct_id','N/A')}  |  Phase: {result.get('trial_phase','?')}  |  Sponsor: {result.get('sponsor','?')[:38]}")
    if result.get("trial_sites"):
        print(f"  📍 Sites: {', '.join(result['trial_sites'][:3])}")
    print(f"  🔗 {result.get('trial_url','')}")

    print(f"\n  📝 Summary:")
    summary = result.get("match_summary", "")
    for line in [summary[i:i+62] for i in range(0, len(summary), 62)]:
        print(f"     {line}")

    criteria = result.get("criteria_results", [])
    if criteria:
        print(f"\n  📊 Criteria breakdown ({len(criteria)} checked):")
        for c in criteria[:8]:
            icon = verdict_icon(c["verdict"])
            text = c["criterion_text"][:55]
            print(f"     {icon}  {text}")
            if c.get("reasoning"):
                print(f"        └─ {c['reasoning'][:70]}")
        if len(criteria) > 8:
            print(f"        ... and {len(criteria)-8} more criteria checked")

    missing = result.get("missing_data_requests", [])
    if missing:
        print(f"\n  🔬 Missing data — order these to confirm eligibility:")
        for m in missing:
            print(f"     → {m}")

print("\n" + "█"*65)
print("  CLINICAL TRIAL MATCHING RESULTS  [powered by Grok-3]")
print(f"  Patient: {patient_profile['demographics']['age']}y {patient_profile['demographics']['sex']}  |  Dx: {patient_profile['primary_diagnosis']['name']}")
print("█"*65)

if not eligible:
    print("\n  ⚠️  No eligible trials found in this search batch.")
    print("  Try adjusting search terms or expanding MAX_TRIALS in Cell 3.")
else:
    for i, match in enumerate(eligible[:TOP_K], 1):
        print_match_card(match, i)

print(f"\n{'─'*65}")
print(f"  Disqualified: {len(ineligible)}  |  Eligible: {len(eligible)}")
print(f"  Showing top {min(TOP_K, len(eligible))} of {len(eligible)} eligible matches")
print(f"{'─'*65}")




█████████████████████████████████████████████████████████████████
  CLINICAL TRIAL MATCHING RESULTS  [powered by Grok-3]
  Patient: 61y F  |  Dx: Stage IIIB non-small cell lung cancer
█████████████████████████████████████████████████████████████████

  MATCH #1  |  Score: 76%  [███████░░░]
  📋 Osimertinib With or Without Bevacizumab as Initial Treatment for Patients With EGFR-Mutant Lung Cancer
  🔑 NCT04181060  |  Phase: PHASE3  |  Sponsor: National Cancer Institute (NCI)
  📍 Sites: Anchorage, United States
  🔗 https://clinicaltrials.gov/study/NCT04181060

  📝 Summary:
     The patient appears to be a good candidate for this trial, wit
     h a confirmed NSCLC diagnosis, advanced disease, and an EGFR e
     xon 19 deletion. However, additional information is needed to 
     confirm eligibility, including measurable disease status and a
     bsence of hemoptysis and significant proteinuria.

  📊 Criteria breakdown (11 checked):
     ✅  NSCLC diagnosis
        └─ Patient has a pathologi

In [26]:
# CELL 12 — Missing data action plan

# Tests/data that would unlock more trial matches for this patient

all_missing = {}
for result in eligible + ineligible:
    for item in result.get("missing_data_requests", []):
        if item not in all_missing:
            all_missing[item] = []
        all_missing[item].append(result.get("trial_nct_id", ""))

if all_missing:
    print("\n" + "="*65)
    print("  🔬 MISSING DATA ACTION PLAN")
    print("  Order these tests to potentially unlock more trial matches:")
    print("="*65)
    for test, trials in sorted(all_missing.items(), key=lambda x: -len(x[1])):
        print(f"\n  → {test}")
        print(f"     Relevant to: {', '.join(trials[:4])}")
    print()
else:
    print("\n✅ No critical missing data identified.")


  🔬 MISSING DATA ACTION PLAN
  Order these tests to potentially unlock more trial matches:

  → Cardiovascular evaluation
     Relevant to: NCT07003490, NCT06788912

  → RECIST v1.1 assessment
     Relevant to: NCT06734182, NCT07288034

  → Total bilirubin level
     Relevant to: NCT07288034, NCT05713994

  → Surgical evaluation
     Relevant to: NCT06788912, NCT04406831

  → Baseline measurements of sites of disease
     Relevant to: NCT04181060

  → Hemoptysis assessment
     Relevant to: NCT04181060

  → Urinalysis results
     Relevant to: NCT04181060

  → Patient's willingness to sign informed consent
     Relevant to: NCT06041776

  → Brain imaging results to confirm absence of metastasis
     Relevant to: NCT06041776

  → Postoperative wound healing status and time since surgery
     Relevant to: NCT06041776

  → Serum pregnancy test results for female patients
     Relevant to: NCT06041776

  → Patient's medical history regarding other malignancies
     Relevant to: NCT0604177

In [28]:
# CELL 13 — Export results to JSON


from datetime import datetime

output = {
    "generated_at"   : datetime.now().isoformat(),
    "model_used"     : {"extraction": MODEL_FAST, "reasoning": MODEL_REASONING},
    "patient_summary": {
        "age"      : patient_profile["demographics"]["age"],
        "sex"      : patient_profile["demographics"]["sex"],
        "diagnosis": patient_profile["primary_diagnosis"]["name"],
        "ecog"     : patient_profile["ecog_status"]
    },
    "search_stats": {
        "trials_fetched"     : len(parsed_trials),
        "after_pre_filter"   : len(pre_filtered),
        "after_llm_reasoning": len(all_results),
        "eligible"           : len(eligible),
        "ineligible"         : len(ineligible)
    },
    "top_matches" : eligible[:TOP_K],
    "missing_data": all_missing,
    "all_results" : all_results
}

output_path = "trial_matching_results_grok.json"
with open(output_path, "w") as f:
    json.dump(output, f, indent=2, default=str)

print(f"Results saved to: {output_path}")

try:
    from google.colab import files
    files.download(output_path)
    print("Download started")
except Exception:
    print(f"   (Not in Colab — file saved locally as {output_path})")



Results saved to: trial_matching_results_grok.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started


In [31]:
# CELL 14 — Summary table


rows = []
for r in all_results:
    criteria = r.get("criteria_results", [])
    rows.append({
        "NCT ID"    : r.get("trial_nct_id", ""),
        "Title"     : r.get("trial_title", "")[:45],
        "Phase"     : r.get("trial_phase", ""),
        "Score"     : f"{r['match_score']:.0%}",
        "PASS"      : sum(1 for c in criteria if c["verdict"] == "PASS"),
        "FAIL"      : sum(1 for c in criteria if c["verdict"] == "FAIL"),
        "UNCERTAIN" : sum(1 for c in criteria if c["verdict"] == "UNCERTAIN"),
        "MISSING"   : sum(1 for c in criteria if c["verdict"] == "MISSING_DATA"),
        "Eligible"  : "✅" if r["match_score"] > 0 else "❌"
    })

df = pd.DataFrame(rows).sort_values("Score", ascending=False)
print("\n Full results table:\n")
print(df.to_string(index=False))



 Full results table:

     NCT ID                                         Title  Phase Score  PASS  FAIL  UNCERTAIN  MISSING Eligible
NCT04181060 Osimertinib With or Without Bevacizumab as In PHASE3   76%     8     0          3        0        ✅
NCT06041776 Adjuvant Befotertinib in Stage IB-IIIB Non-sm PHASE3   37%     5     0          2        5        ✅
NCT07003490 Radical Resection With Contralateral Lymph No     NA   17%     3     0          6        0        ✅
NCT06734182 Neoadjuvant Envafolimab Plus Disitamab Vedoti PHASE2    0%     4     1          1        3        ❌
NCT04406831 The Role of MicroRNA in the Diagnosis, Progno    N/A    0%     1     4          1        1        ❌
NCT05665361 Palbociclib and Sasanlimab for the Treatment  PHASE1    0%     5     1          0        0        ❌
NCT04105270 RMT in Combination With Durvalumab + Chemo in PHASE2    0%     2     2          3        2        ❌
NCT05661786 Clinical Outcomes of HBeAg-negative CHB Patie    N/A    0%     1     

In [30]:
# CELL 15 — Try a different patient (optional)

# Uncomment one profile below and re-run from Cell 5

# --- Patient B: Breast cancer ---
# PATIENT_EHR = """
# 45-year-old female. Stage II HER2-positive breast cancer, diagnosed 3 months ago.
# ER negative, PR negative, HER2 3+ by IHC. BRCA1/2 negative.
# ECOG 0. No prior systemic therapy. No significant comorbidities.
# Current meds: None. Labs: normal CBC, LFTs, renal function.
# Location: Houston, TX, USA.
# """

# --- Patient C: Colorectal cancer ---
# PATIENT_EHR = """
# 72-year-old male. Stage IV colorectal adenocarcinoma with liver metastases.
# KRAS wild-type. MSI-H / dMMR confirmed. ECOG 2.
# Prior therapy: FOLFOX x6 cycles (completed 2 months ago), bevacizumab.
# Comorbidities: Atrial fibrillation on warfarin, CKD stage 3 (eGFR 38).
# Location: Chicago, IL, USA.
# """

# --- Patient D: Healthy volunteer ---
# PATIENT_EHR = """
# 28-year-old male, healthy volunteer. No chronic conditions. No medications.
# BMI 23. Non-smoker. No known allergies. ECOG 0.
# Location: New York, NY, USA.
# """

print("✅ To test a new patient:")
print("   1. Uncomment one profile above (or write your own)")
print("   2. Re-run from Cell 5 downward")
print("   3. Everything else runs automatically")

✅ To test a new patient:
   1. Uncomment one profile above (or write your own)
   2. Re-run from Cell 5 downward
   3. Everything else runs automatically
